In [5]:
import time
from typing import Dict, Any

In [6]:
# A mock pricing database mapping model names to their cost per 1,000,000 tokens in USD
# (Prices represent realistic 2026 developer platform structures)
MODEL_PRICING_REGISTRY = {
    "gpt-4o-mini": {
        "input_cost_per_m": 0.15,   # $0.15 per million tokens
        "output_cost_per_m": 0.60,  # $0.60 per million tokens
    },
    "gpt-4o": {
        "input_cost_per_m": 2.50,   # $2.50 per million tokens
        "output_cost_per_m": 10.00, # $10.00 per million tokens
    }
}

def calculate_api_cost(model_name: str, input_tokens: int, output_tokens: int) -> float:
    """Calculates the exact monetary cost of an LLM transaction."""
    if model_name not in MODEL_PRICING_REGISTRY:
        return 0.0
    
    rates = MODEL_PRICING_REGISTRY[model_name]
    input_cost = (input_tokens / 1_000_000) * rates["input_cost_per_m"]
    output_cost = (output_tokens / 1_000_000) * rates["output_cost_per_m"]
    
    return input_cost + output_cost

def execute_tracked_llm_call(model_name: str, system_prompt: str, user_input: str) -> Dict[str, Any]:
    """
    Executes an LLM call while explicitly measuring latency and capturing
    token usage details to calculate transactional costs.
    """
    print(f"\n[API] Dispatching request to model: '{model_name}'...")
    
    # Start the execution timer
    start_time = time.time()
    
    # --- Simulated Network Request / Response Cycle ---
    # In a real application, this would be: response = openai_client.chat.completions.create(...)
    time.sleep(1.2)  # Simulating a 1.2 second network and generation delay
    
    mock_response_text = "Analysis complete. The anomalous signal indicates a high-frequency micro-fissure."
    
    # Simulating explicit token usage metadata returned by the provider's API payload
    # Let's assume input text was roughly 40 tokens and output text was 15 tokens
    mock_usage_metadata = {
        "prompt_tokens": 42,
        "completion_tokens": 15,
        "total_tokens": 57
    }
    # ---------------------------------------------------
    
    # Stop the timer immediately after the payload is received
    end_time = time.time()
    total_latency = end_time - start_time
    
    # Extract structural token usage numbers
    prompt_tokens = mock_usage_metadata["prompt_tokens"]
    completion_tokens = mock_usage_metadata["completion_tokens"]
    
    # Calculate costs using our utility function
    transaction_cost = calculate_api_cost(model_name, prompt_tokens, completion_tokens)
    
    # Compile performance trace payload
    telemetry_report = {
        "model": model_name,
        "output_text": mock_response_text,
        "metrics": {
            "latency_seconds": round(total_latency, 3),
            "tokens_input": prompt_tokens,
            "tokens_output": completion_tokens,
            "tokens_total": mock_usage_metadata["total_tokens"],
            "estimated_cost_usd": float(f"{transaction_cost:.7f}") # High precision float
        }
    }
    
    return telemetry_report

In [7]:
# --- Test Case 1: Low-Cost Efficient Model ---
cheap_run = execute_tracked_llm_call(
    model_name="gpt-4o-mini",
    system_prompt="You are a quick triage assistant.",
    user_input="Analyze logs from server rack 4."
)

print("Telemetry Output (Mini):")
print(f"  • Latency: {cheap_run['metrics']['latency_seconds']} seconds")
print(f"  • Token Breakdown: In: {cheap_run['metrics']['tokens_input']} | Out: {cheap_run['metrics']['tokens_output']}")
print(f"  • Computed Cost: ${cheap_run['metrics']['estimated_cost_usd']}")


[API] Dispatching request to model: 'gpt-4o-mini'...
Telemetry Output (Mini):
  • Latency: 1.2 seconds
  • Token Breakdown: In: 42 | Out: 15
  • Computed Cost: $1.53e-05


In [8]:
# --- Test Case 2: High-Performance Premium Model ---
premium_run = execute_tracked_llm_call(
    model_name="gpt-4o",
    system_prompt="You are an expert deep-diagnostic intelligence system.",
    user_input="Analyze logs from server rack 4."
)

print("\nTelemetry Output (Premium):")
print(f"  • Latency: {premium_run['metrics']['latency_seconds']} seconds")
print(f"  • Token Breakdown: In: {premium_run['metrics']['tokens_input']} | Out: {premium_run['metrics']['tokens_output']}")
print(f"  • Computed Cost: ${premium_run['metrics']['estimated_cost_usd']}")


[API] Dispatching request to model: 'gpt-4o'...

Telemetry Output (Premium):
  • Latency: 1.2 seconds
  • Token Breakdown: In: 42 | Out: 15
  • Computed Cost: $0.000255
